In [2]:
!pip install PyMuPDF langchain-text-splitters langchain-ollama langchain-community langchain-core chromadb


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.9/24.9 MB 44.4 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 72.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 69.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 60.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 58.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 59.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.5/21.5 MB 68.6 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 613.9/613.9 kB 48.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 74.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.1/17.1 MB 67.6 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.7/6.7 MB 61.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 58.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [ ]:
import fitz
import os
import shutil
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_ollama import OllamaEmbeddings, ChatOllama
from langchain_community.vectorstores import Chroma
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

# ==========================================
# 階段一：資料處理與向量庫建立 (Data Ingestion)
# ==========================================


model = 'qwen3.5:122b'


# 1. Load PDF (讀取 PDF)
import os

# path_for_local = "/Users/max/Desktop/antigravity/DanceLight--LED-AI-Chatbot-and-attribute-searching_v2"
# just 
#path_for_ollama = "2025舞光LED21st(單頁水印可搜尋).pdf"
#os.chdir('/Users/max/Desktop/antigravity/DanceLight--LED-AI-Chatbot-and-attribute-searching_v2')
doc = fitz.open("2025舞光LED21st(單頁水印可搜尋).pdf")
text = "".join([page.get_text() for page in doc])

# 2. Chunking (切分文本)
text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=100)
chunks = text_splitter.split_text(text)

# 3. Embedding Model (設定本地端 EmbeddingGemma 模型)
# 確保你已在終端機執行過: ollama pull embeddinggemma
embeddings = OllamaEmbeddings(model="embeddinggemma")

# 4. 向量資料庫處理
persist_dir = "./test_gemma_vector_db"

# Clear chromadb cache to prevent sqlite read-only errors in Jupyter
import chromadb
chromadb.api.client.SharedSystemClient.clear_system_cache()

# 每次執行都先清空舊的資料庫，確保資料是最新重切的
if os.path.exists(persist_dir):
    shutil.rmtree(persist_dir)
    print("已刪除舊的向量庫，準備以 EmbeddingGemma 模型重建...")

# 建立並儲存新的向量庫
vector_db = Chroma.from_texts(
    chunks, 
    embeddings, 
    persist_directory=persist_dir
)

print(f"成功！已建立資料庫。總共切分了 {len(chunks)} 個區塊。")

# ==========================================
# 階段二：檢索與大語言模型問答 (Retrieval & Generation)
# ==========================================

# 1. Setup Retriever (設定檢索器)
retriever = vector_db.as_retriever(search_kwargs={"k": 5})

# 2. Define the Local LLM (設定 Llama 4 Scout)
# 確保你已在終端機執行過: ollama pull llama4:scout
llm = ChatOllama(
    model=model
    temperature=0  # 保持 0 讓回答精準，不隨便發揮創意
)

# 3. Create the Prompt (設定專家 Prompt)
template = """
### 角色定位
你是一位專業的「舞光 (DanceLight)」LED 照明產品顧問。你的任務是根據提供的目錄內容，為客戶提供準確、專業且親切的產品建議。

### 檢索到的目錄內容 (Context)
{context}

### 任務指令
1. **分析需求**：判斷使用者是尋找「特定產品名稱」（如：邱比特軌道燈）還是針對「特定場所」的需求（如：浴室、客廳）並且根據使用場景來推薦產品。
2. **規格導向**：在推薦產品時，請務必從 context 中提取以下技術屬性（若有標示）：
   - **瓦數 (W)**
   - **色溫 (K)**：如暖白光 (3000K)、自然光 (4000K) 或白光 (6000K)。
   - **演色性 (Ra)**：特別提到 Ra > 90 的高演色性產品。
   - **特殊功能**：如 IP65 防水等級、防眩光、低頻閃等。
3. **排版方式**：請使用「列點」格式，讓產品規格一目了然。

### 回答限制
- **範疇限制**：僅根據上方提供的「檢索到的目錄內容」進行回答。如果 context 中沒有相關資訊，請禮貌地回答：「抱歉，目前的產品目錄中沒有提到相關細節，建議您可以詢問其他類似款式。」
- **嚴禁胡說**：不可自行編造產品型號、規格或價格。
- **語言一致性**：一律使用「繁體中文」回答。

---
使用者問題：{question}
顧問建議：
"""
prompt = ChatPromptTemplate.from_template(template)

# 4. Build the Chain using LCEL (組裝 RAG 流程)
qa_chain = (
    {"context": retriever, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

# 5. Ask a question (提出問題)
print("\n ...]\n")
response = qa_chain.invoke("天花板不需開槽的排燈是哪些型號")
print(response)

成功！已建立資料庫。總共切分了 441 個區塊。

[Llama 4 Scout 思考中...]

您好，我是舞光 (DanceLight) 的 LED 照明產品顧問。根據您提供的目錄內容，針對「天花板不需開槽」的安裝需求，以下為您整理出符合條件的產品系列及其技術規格：

### 1. 拉斐爾超薄磁吸軌道系列
此系列在目錄中明確標示「天花板無須開槽即可安裝」，適合追求無主燈設計且不希望破壞天花板結構的客戶。

*   **產品型號**：
    *   D-UTMTTR8N (8W / 4000K)
    *   D-UTMTTR8W (8W / 3000K)
    *   D-UTMTTR15N (15W / 4000K)
    *   D-UTMTTR15W (15W / 3000K)
*   **消耗電力**：8W 或 15W
*   **色溫**：3000K (暖白光) 或 4000K (自然光)
*   **演色性 (Ra)**：Ra ≥ 90 (高演色性，真實呈現顏色)
*   **光通量**：
    *   8W 系列：550LM (3000K) / 600LM (4000K)
    *   15W 系列：1250LM (3000K) / 1400LM (4000K)
*   **特殊功能**：
    *   低色容差 (SDCM < 3.5)，光色無色差
    *   光束角 36°
    *   需外接驅動器 (輸入電壓 DC24V)
*   **燈體材質**：壓鑄鋁、PMMA 罩

### 2. 鋁槽燈系列 (吸頂式安裝選項)
此系列為線性排燈，目錄中提供「吸頂式」安裝選項，通常無需進行天花板開槽作業（亦可選擇崁入式，視需求而定）。

*   **產品型號**：
    *   D-0809AT 系列 (適用 2835 軟條燈 120P)
    *   LED-1010AT 系列 (適用 COB 軟條燈、2835 軟條燈 120P)
    *   LED-1212AT 系列 (適用 COB 軟條燈、2835 軟條燈 240P/120P)
    *   D-1507AT 系列 (適用 COB 軟條燈)
*   **長度選項**：1 米、2 米、3 米
*   **安裝方式**：提供「吸頂式」與「崁入式」兩種選擇（吸頂式無需開槽

In [6]:
# Use .stream() instead of .invoke()
for chunk in qa_chain.stream("舞蹈教室適合的 LED 燈有哪些？"):
    print(chunk, end="", flush=True)
print("\n")

您好！歡迎諮詢「舞光 (DanceLight)」LED 照明產品顧問。針對您詢問的「舞蹈教室」照明需求，這類場所通常屬於商業空間，需要高亮度、高演色性以確保舞者能清楚看見動作與服裝細節，同時光線需均勻無眩光。

根據目前的產品目錄內容，為您篩選出以下適合商業空間與高品質照明需求的產品建議：

### 1. 商業空間專用軌道/投射燈系列 (高演色性首選)
此系列明確標示為「商業空間照明」，具備高演色性與高亮度，非常適合舞蹈教室作為主要照明，能還原真實色彩。
*   **產品型號**：LED-TR30WFLR1 / LED-TR45WFLR1 (時尚白) 或 LED-TR30DFLBKR1 / LED-TR45DFLBKR1 (貴族黑)
*   **瓦數 (W)**：30W 或 45W
*   **色溫 (K)**：
    *   3000K (暖白光)
    *   4000K (自然光)
    *   5000K (白光)
*   **光通量**：2050LM (30W) / 3100-3600LM (45W)
*   **演色性 (Ra)**：Ra ≥ 90，R9 > 50 (高演色性，適合色彩辨識)
*   **發光角度**：30° (重點照明) / 350° 或 355° (廣角照明)
*   **備註**：此系列為特價限量商品，適合需要高品質光環境的場所。

### 2. COB 室內專用 DC 24V 軟條燈模組 (間接照明/氛圍)
若舞蹈教室需要設計間接照明（如天花板燈槽、鏡櫃周邊），此系列能提供無光斑的均勻光線，且具備博物館級光品質。
*   **產品型號**：LED-35CO24V10D / LED-35CO24V10N / LED-35CO24V10W
*   **瓦數 (W)**：10W / 米
*   **色溫 (K)**：
    *   6500K (白光)
    *   4000K (自然光)
    *   3000K (暖白光)
*   **光通量**：800LM / 米
*   **演色性 (Ra)**：Ra ≥ 90 (超高演色性)
*   **特殊功能**：
    *   完美超薄，無光斑
    *   可彎最小直徑為 50mm (勿對折)
    *   內附固定夾，可加強固定避免脫落
    *   每 4 公分

In [11]:
from langchain_community.vectorstores import Chroma
from langchain_ollama import OllamaEmbeddings, ChatOllama
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

model = 'qwen3.5:122b'

# 1. Load EXISTING Vector DB (No PDF parsing here!)
embeddings = OllamaEmbeddings(model="embeddinggemma")
vector_db = Chroma(
    persist_directory="./test_gemma_vector_db", 
    embedding_function=embeddings
)

retriever = vector_db.as_retriever(search_kwargs={"k": 5})

# 2. Define LLM (Ensure GPU acceleration is active)
llm = ChatOllama(
    model=model, 
    temperature=0,
    num_gpu=99,      # Forces Ollama to put all layers into VRAM
    keep_alive="30m"  # Keeps the model in VRAM indefinitely
)

# 3. Create the Prompt (Same as your original)
template = """
### 角色定位
你是一位專業的「舞光 (DanceLight)」LED 照明產品顧問。你的任務是根據提供的目錄內容，為客戶提供準確、專業且親切的產品建議。

### 檢索到的目錄內容 (Context)
{context}

### 任務指令
1. **分析需求**：判斷使用者是尋找「特定產品名稱」還是針對「特定場所」的需求，並根據使用場景來推薦產品。
2. **規格導向**：在推薦產品時，請提取以下屬性（若有標示）：瓦數 (W)、色溫 (K)、演色性 (Ra)、特殊功能。
3. **排版方式**：請使用「列點」格式。

### 回答限制
- 僅根據上方提供的「檢索到的目錄內容」進行回答。
- 嚴禁自行編造產品型號、規格或價格。
- 一律使用「繁體中文」回答。

---
使用者問題：{question}
顧問建議：
"""
prompt = ChatPromptTemplate.from_template(template)

# 4. Build RAG Chain
from langchain_core.runnables import RunnableParallel

rag_chain_from_docs = (
    RunnablePassthrough.assign(context=(lambda x: x["context"]))
    | prompt
    | llm
    | StrOutputParser()
)

qa_chain = RunnableParallel(
    {"context": retriever, "question": RunnablePassthrough()}
).assign(answer=rag_chain_from_docs)

# 5. Execute with Streaming
print(f"\n[{model} 檢索與思考中...]\n")
question = "浴室適合的 LED 燈有哪些？"

# We stream just the answer part to avoid printing the dictionary structure repeatedly
# First, fetch the context manually
docs = retriever.invoke(question)

# Create a clean stream from the doc chain
print("\n--- LLM 思考與回答 ---\n")
for chunk in rag_chain_from_docs.stream({"context": docs, "question": question}):
    print(chunk, end="", flush=True)

print("\n--- LLM 回答 ---\n")
print(response["answer"])
print("\n")



[qwen3.5:122b 檢索與思考中...]


--- LLM 思考與回答 ---

您好！我是舞光 (DanceLight) 的 LED 照明產品顧問。針對您詢問「浴室適合的 LED 燈」，根據目錄內容，浴室環境潮濕，建議選擇具有明確防水等級的產品。

以下為您推薦目錄中明確標示適合浴室使用的產品系列：

*   **尼莫防水崁燈系列**
    *   **推薦原因**：目錄中明確標示「整燈防水 IP66，適裝浴室、騎樓」，是針對潮濕環境設計的最佳選擇。
    *   **產品型號**：
        *   OD-15DON16D (白光)
        *   OD-15DON16W (黃光)
    *   **規格參數**：
        *   **消耗電力**：16W
        *   **色溫**：6000K (白光) / 3000K (黃光)
        *   **光通量**：1440LM (6000K) / 1340LM (3000K)
        *   **演色性**：≧80
        *   **輸入電壓**：100-240V (全電壓)
        *   **發光角度**：160°
        *   **燈體材質**：散熱塑、PC 罩
    *   **特殊功能**：
        *   驅動器內藏，輕巧方便安裝。
        *   滿版晶片，均勻發光無暗區。
        *   採用超音波結合，防水堵線有效防水。

*   **商業空間照明 防水崁燈系列**
    *   **推薦原因**：此系列歸類於「防水崁燈系列」，具備防水特性，可作為浴室照明的替代方案。
    *   **產品型號範例**：LED-CE12WR3、LED-CE16DR3、LED-CE16WR3 等。
    *   **規格參數**：
        *   **瓦數**：12W / 16W / 30W
        *   **色溫**：3000K (暖白) / 6500K (冷白)
        *   **光通量**：850LM ~ 3300LM (視型號而定)
        *   **燈體材質**：PMMA、烤漆鐵底盤、PC 防觸電罩
    *   **特殊功能**：
   